In [3]:
# Deactivate distracting warnings
import warnings
warnings.filterwarnings("ignore")

In [4]:
from utils.data_processing import *
from utils.visualization import load_data, plot_event_count, plot_kde, plot_goal_positions
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import pandas as pd

In [49]:
matches = [
    {
        "match_id": "J03WOH",
        "path": "./idsse-data/DFL-MAT-J03WOH/",
        "file_name_pos": "DFL_04_03_positions_raw_observed_DFL-COM-000002_DFL-MAT-J03WOH.xml",
        "file_name_infos": "DFL_02_01_matchinformation_DFL-COM-000002_DFL-MAT-J03WOH.xml",
        "file_name_events": "DFL_03_02_events_raw_DFL-COM-000002_DFL-MAT-J03WOH.xml",
    },
    {
        "match_id": "J03WMX",
        "path": "./idsse-data/DFL-MAT-J03WMX/",
        "file_name_pos": "DFL_04_03_positions_raw_observed_DFL-COM-000001_DFL-MAT-J03WMX.xml",
        "file_name_infos": "DFL_02_01_matchinformation_DFL-COM-000001_DFL-MAT-J03WMX.xml",
        "file_name_events": "DFL_03_02_events_raw_DFL-COM-000001_DFL-MAT-J03WMX.xml", 
    },
    {
        "match_id": "J03WPY",
        "path": "./idsse-data/DFL-MAT-J03WPY/",
        "file_name_pos": "DFL_04_03_positions_raw_observed_DFL-COM-000002_DFL-MAT-J03WPY.xml",
        "file_name_infos": "DFL_02_01_matchinformation_DFL-COM-000002_DFL-MAT-J03WPY.xml",
        "file_name_events": "DFL_03_02_events_raw_DFL-COM-000002_DFL-MAT-J03WPY.xml", 
    },
    {
        "match_id": "J03WR9",
        "path": "./idsse-data/DFL-MAT-J03WR9/",
        "file_name_pos": "DFL_04_03_positions_raw_observed_DFL-COM-000002_DFL-MAT-J03WR9.xml",
        "file_name_infos": "DFL_02_01_matchinformation_DFL-COM-000002_DFL-MAT-J03WR9.xml",
        "file_name_events": "DFL_03_02_events_raw_DFL-COM-000002_DFL-MAT-J03WR9.xml", 
    }
]

In [44]:
DEFENDERS = ['LV', 'IVL', 'IVR', 'RV']
def get_first_half_players(valid_counts, positions):
    """
    valid_counts: 각 컬럼별 정상 데이터 개수가 담긴 리스트나 배열 (예: 40개 요소)
    positions: 선수 인덱스를 인덱스로 갖는 포지션 Series (예: 20개 요소)
    """
    starting_players = {}
    
    # 총 선수 명수 계산 (컬럼 수 / 2)
    num_players = len(valid_counts) // 2
    
    for i in range(num_players):
        # 선수의 x좌표 컬럼 인덱스 계산 (0, 2, 4, 6...)
        # y좌표는 2*i + 1이지만, x좌표 하나만 검사해도 출전 여부 파악 가능
        x_col_idx = i * 2 
        
        # 해당 선수의 좌표 데이터가 1개라도 존재한다면(0보다 크다면) 출전한 선수로 간주
        if valid_counts[x_col_idx] > 0:
            # 포지션 Series에서 해당 인덱스(i)의 포지션 값을 가져옴
            pos = positions[i]
            
            # 딕셔너리에 추가
            starting_players[i] = pos
            
    return starting_players

def filter_defenders(result, target_positions = ['LV', 'IVL', 'IVR', 'RV']):
    target_columns = []
    new_column_names = []

    for player_idx, pos in result.items():
        if pos in target_positions:
            # x좌표 컬럼: 인덱스 * 2, y좌표 컬럼: 인덱스 * 2 + 1
            x_col = player_idx * 2
            y_col = player_idx * 2 + 1
            
            target_columns.extend([x_col, y_col])
            new_column_names.extend([f'{pos}_x', f'{pos}_y'])
    return target_columns, new_column_names

def get_backline_mean_x(df, defenders=DEFENDERS):
    x_cols = [f"{p}_x" for p in defenders]
    return df[x_cols].mean(axis=1).mean()

def infer_attacking_direction_from_defenders(home_df, away_df, defenders=DEFENDERS):
    """
    반환값:
        home_attack_dir = +1 이면 home이 -x -> +x 방향 공격
        home_attack_dir = -1 이면 home이 +x -> -x 방향 공격

        away_attack_dir도 동일
    """
    home_mean_x = get_backline_mean_x(home_df, defenders)
    away_mean_x = get_backline_mean_x(away_df, defenders)

    print("home backline mean x:", home_mean_x)
    print("away backline mean x:", away_mean_x)

    if home_mean_x > away_mean_x:
        # 홈팀 수비라인이 +x 쪽에 있음
        # 홈팀은 +x 골대를 지키므로 공격은 -x 방향
        home_attack_dir = -1
        away_attack_dir = +1
    else:
        # 홈팀 수비라인이 -x 쪽에 있음
        # 홈팀은 -x 골대를 지키므로 공격은 +x 방향
        home_attack_dir = +1
        away_attack_dir = -1

    return home_attack_dir, away_attack_dir

def normalize_to_positive_attack(df, attack_dir, defenders=DEFENDERS):
    """
    모든 팀이 +x 방향으로 공격하는 것처럼 좌표를 정규화한다.

    attack_dir:
        +1 = 이미 +x 방향 공격
        -1 = -x 방향으로 공격하므로 x,y 모두 반전
    """
    df = df.copy()

    if attack_dir == -1:
        for p in defenders:
            df[f"{p}_x"] = -df[f"{p}_x"]
            df[f"{p}_y"] = -df[f"{p}_y"]

    return df

In [50]:
all_def_dfs = []

for match in matches:
    xy_objects, events, pitch, possession, ballstatus, teamsheets = load_data(match["path"], match["file_name_pos"], match["file_name_infos"], match["file_name_events"])
    Home_first = xy_objects["firstHalf"]["Home"].xy 
    Away_first = xy_objects["firstHalf"]["Away"].xy 
    Possession_first = possession["firstHalf"].code
    Ballstatus_first = ballstatus["firstHalf"].code
    print(match["match_id"])
    non_nan_counts = np.sum(~np.isnan(Home_first), axis=0)
    non_nan_counts_2 = np.sum(~np.isnan(Away_first), axis=0)
    result_Home = get_first_half_players(non_nan_counts, teamsheets['Home']['position'])
    result_Away = get_first_half_players(non_nan_counts_2, teamsheets['Away']['position'])
    Home_first_df = pd.DataFrame(Home_first)
    Away_first_df = pd.DataFrame(Away_first)
    Possession_first_df = pd.DataFrame(Possession_first)
    Ballstatus_first_df = pd.DataFrame(ballstatus["firstHalf"].code)
    
    target_col_home, new_col_home = filter_defenders(result_Home)
    target_col_away, new_col_away = filter_defenders(result_Away)

    print(new_col_away)
    filtered_df_Home = Home_first_df[target_col_home]
    filtered_df_Home.columns = new_col_home

    filtered_df_Away = Away_first_df[target_col_away]
    filtered_df_Away.columns = new_col_away

    filtered_df_Home['possession'] = Possession_first_df.values
    filtered_df_Away['possession'] = Possession_first_df.values
    filtered_df_Home['ballstatus'] = Ballstatus_first_df.values
    filtered_df_Away['ballstatus'] = Ballstatus_first_df.values

    home_attack_dir, away_attack_dir = infer_attacking_direction_from_defenders(
        filtered_df_Home,
        filtered_df_Away
    )

    home_df_norm = normalize_to_positive_attack(filtered_df_Home, home_attack_dir)
    away_df_norm = normalize_to_positive_attack(filtered_df_Away, away_attack_dir)

    home_defense_df = home_df_norm[(home_df_norm["possession"] == 2)&(home_df_norm["ballstatus"]==1)].copy()
    away_defense_df = away_df_norm[(away_df_norm["possession"] == 1)&(away_df_norm["ballstatus"]==1)].copy()
    
    print(home_defense_df.head())
    print(away_defense_df.tail())

    all_def_dfs.append(home_defense_df)
    all_def_dfs.append(away_defense_df)


    

J03WOH
['LV_x', 'LV_y', 'IVR_x', 'IVR_y', 'RV_x', 'RV_y', 'IVL_x', 'IVL_y']
home backline mean x: 12.09367520598064
away backline mean x: -14.827426502362298
    RV_x   RV_y  IVR_x  IVR_y  IVL_x  IVL_y   LV_x   LV_y  possession  \
0 -12.53 -20.32 -16.45  -6.50 -17.32   5.08 -13.81  14.50         2.0   
1 -12.55 -20.34 -16.47  -6.56 -17.29   5.07 -13.82  14.52         2.0   
2 -12.59 -20.35 -16.51  -6.60 -17.27   5.06 -13.84  14.54         2.0   
3 -12.62 -20.38 -16.54  -6.65 -17.25   5.05 -13.85  14.56         2.0   
4 -12.65 -20.41 -16.57  -6.70 -17.22   5.04 -13.87  14.57         2.0   

   ballstatus  
0         1.0  
1         1.0  
2         1.0  
3         1.0  
4         1.0  
        LV_x  LV_y  IVR_x  IVR_y   RV_x  RV_y  IVL_x  IVL_y  possession  \
68654 -39.45  3.58 -41.26  -0.68 -38.63  0.31 -42.83  -4.76         1.0   
68655 -39.39  3.60 -41.23  -0.70 -38.59  0.26 -42.81  -4.73         1.0   
68656 -39.34  3.61 -41.20  -0.74 -38.56  0.20 -42.79  -4.69         1.0   
68657 -

len(xy_objects["firstHalf"]["Home"][0])=40

In [ ]:

result_Home = get_first_half_players(non_nan_counts, teamsheets['Home']['position'])
result_Away = get_first_half_players(non_nan_counts_2, teamsheets['Away']['position'])

print("Home 전반전 출전 선수 (인덱스: 포지션):")
for idx, pos in result_Home.items():
    print(f"Index {idx:2d} : {pos}")
print("Away 전반전 출전 선수 (인덱스: 포지션):")
for idx, pos in result_Away.items():
    print(f"Index {idx:2d} : {pos}")

Home 전반전 출전 선수 (인덱스: 포지션):
Index  2 : STZ
Index  3 : RV
Index  4 : IVR
Index  5 : ORM
Index  6 : DML
Index  7 : DMR
Index  8 : IVL
Index  9 : ZO
Index 11 : TW
Index 12 : LV
Index 15 : OLM
Away 전반전 출전 선수 (인덱스: 포지션):
Index  0 : LV
Index  1 : DMR
Index  2 : OLM
Index  3 : IVR
Index  5 : ORM
Index  6 : TW
Index  8 : DML
Index 11 : STZ
Index 12 : RV
Index 13 : ZO
Index 16 : IVL


# 1. 홈, 원정팀 수비수 좌표 및 possession df 만들기

# 2. 공격 방향 통일

In [51]:
def get_lateral_diffs(frame):
    """
    frame example:
    {
        "LV":  {"x": ..., "y": ...},
        "IVL": {"x": ..., "y": ...},
        "IVR": {"x": ..., "y": ...},
        "RV":  {"x": ..., "y": ...}
    }
    """
    sorted_players = sorted(frame.items(), key=lambda item: item[1]["y"])

    base_y = sorted_players[0][1]["y"]

    lateral_diffs = {
        player: coords["y"] - base_y
        for player, coords in sorted_players
    }

    return lateral_diffs

# 3. Vertical gap 구하기

In [52]:

def get_lateral_gaps_from_row(row, defenders=DEFENDERS):
    """
    한 프레임에서 수비수 4명을 y축 기준으로 정렬하고,
    인접 선수 간 y-gap 3개를 계산한다.

    Returns:
        sorted_players: y값 기준 아래 -> 위 순서의 선수 이름
        gaps: [gap_1, gap_2, gap_3]
    """
    player_y = {
        p: row[f"{p}_y"]
        for p in defenders
    }

    sorted_players = sorted(player_y.keys(), key=lambda p: player_y[p])

    sorted_y = np.array(
        [player_y[p] for p in sorted_players],
        dtype=float
    )

    gaps = np.diff(sorted_y)

    return sorted_players, gaps

In [32]:
sorted_players, gaps = get_lateral_gaps_from_row(away_defense_df.iloc[0])

print(sorted_players)
print(gaps)

['LV', 'IVL', 'IVR', 'RV']
[15.18 12.16 10.06]


In [53]:
def add_lateral_gap_features(df, defenders=DEFENDERS):
    """
    각 프레임마다 y축 인접 gap 3개를 추가한다.

    gap_1: 가장 아래쪽 선수와 두 번째 선수 사이
    gap_2: 두 번째 선수와 세 번째 선수 사이
    gap_3: 세 번째 선수와 가장 위쪽 선수 사이
    """
    df = df.copy()

    sorted_player_list = []
    gap_1_list = []
    gap_2_list = []
    gap_3_list = []

    for _, row in df.iterrows():
        sorted_players, gaps = get_lateral_gaps_from_row(row, defenders)

        sorted_player_list.append(sorted_players)
        gap_1_list.append(gaps[0])
        gap_2_list.append(gaps[1])
        gap_3_list.append(gaps[2])

    df["y_order"] = sorted_player_list
    df["lat_gap_1"] = gap_1_list
    df["lat_gap_2"] = gap_2_list
    df["lat_gap_3"] = gap_3_list

    return df

In [54]:
def_dfs = []
for df in all_def_dfs:
  def_df = add_lateral_gap_features(df)
  def_dfs.append(def_df)


In [56]:
def_dfs[0][
    ["y_order", "lat_gap_1", "lat_gap_2", "lat_gap_3"]
].tail()

,y_order,lat_gap_1,lat_gap_2,lat_gap_3
68569,"[IVL, RV, IVR, LV]",3.19,1.95,11.77
68570,"[IVL, RV, IVR, LV]",3.23,1.88,11.71
68571,"[IVL, RV, IVR, LV]",3.29,1.81,11.66
68572,"[IVL, RV, IVR, LV]",3.33,1.74,11.60
68573,"[IVL, RV, IVR, LV]",3.40,1.67,11.54


In [57]:
def fit_gap_thresholds_kmeans(df_list, gap_cols=None, n_clusters=3):
    """
    여러 팀/경기의 defensive dataframe에서 gap 값을 모아서
    Short / Medium / Long 기준 threshold를 KMeans로 학습한다.
    """
    if gap_cols is None:
        gap_cols = ["lat_gap_1", "lat_gap_2", "lat_gap_3"]

    all_gaps = []

    for df in df_list:
        gaps = df[gap_cols].values.reshape(-1)
        gaps = gaps[~np.isnan(gaps)]
        all_gaps.extend(gaps)

    all_gaps = np.array(all_gaps).reshape(-1, 1)

    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=42,
        n_init="auto"
    )
    kmeans.fit(all_gaps)

    centers = np.sort(kmeans.cluster_centers_.flatten())

    thresholds = [
        (centers[i] + centers[i + 1]) / 2
        for i in range(len(centers) - 1)
    ]

    return {
        "model": kmeans,
        "centers": centers,
        "thresholds": thresholds,
        "all_gaps": all_gaps.flatten()
    }

### 3.1 kmeans gap 결과

In [58]:
gap_cluster_result = fit_gap_thresholds_kmeans(
    def_dfs,
    n_clusters=3
)

print("Cluster centers:", gap_cluster_result["centers"])
print("Thresholds:", gap_cluster_result["thresholds"])

Cluster centers: [ 5.35264608 11.43591754 17.92706658]
Thresholds: [np.float64(8.394281812190632), np.float64(14.681492063244793)]


In [59]:
def classify_gap(gap, thresholds):
    """
    gap 하나를 Short / Medium / Long으로 변환한다.

    0 = Short
    1 = Medium
    2 = Long
    """
    t1, t2 = thresholds

    if gap < t1:
        return 0
    elif gap < t2:
        return 1
    else:
        return 2
    
def add_lateral_gap_code(df, thresholds):
    """
    lat_gap_1, lat_gap_2, lat_gap_3을 각각 0/1/2로 변환하고
    morphology code를 생성한다.

    예:
        [0, 2, 1] -> "021"
    """
    df = df.copy()

    gap_cols = ["lat_gap_1", "lat_gap_2", "lat_gap_3"]

    for col in gap_cols:
        df[f"{col}_cls"] = df[col].apply(
            lambda x: classify_gap(x, thresholds)
        )

    df["lat_morphology"] = (
        df["lat_gap_1_cls"].astype(str)
        + df["lat_gap_2_cls"].astype(str)
        + df["lat_gap_3_cls"].astype(str)
    )

    return df

In [60]:
thresholds = gap_cluster_result["thresholds"]

vertical_df = []
for df in def_dfs:
    vdf = add_lateral_gap_code(df, thresholds)
    vertical_df.append(vdf)

In [62]:
vertical_df[0][
    [
        "y_order",
        "lat_gap_1", "lat_gap_2", "lat_gap_3",
        "lat_gap_1_cls", "lat_gap_2_cls", "lat_gap_3_cls",
        "lat_morphology"
    ]
].tail()

,y_order,lat_gap_1,lat_gap_2,lat_gap_3,lat_gap_1_cls,lat_gap_2_cls,lat_gap_3_cls,lat_morphology
68569,"[IVL, RV, IVR, LV]",3.19,1.95,11.77,0,0,1,001
68570,"[IVL, RV, IVR, LV]",3.23,1.88,11.71,0,0,1,001
68571,"[IVL, RV, IVR, LV]",3.29,1.81,11.66,0,0,1,001
68572,"[IVL, RV, IVR, LV]",3.33,1.74,11.60,0,0,1,001
68573,"[IVL, RV, IVR, LV]",3.40,1.67,11.54,0,0,1,001


# 4. Depth delta 구하기

In [63]:
def collect_depth_diffs(df, defenders=DEFENDERS):
    """
    각 프레임에서 가장 뒤쪽 선수 min_x를 기준으로
    4명의 depth_diff를 모두 수집한다.
    """
    all_depth_diffs = []

    for _, row in df.iterrows():
        x_values = np.array(
            [row[f"{p}_x"] for p in defenders],
            dtype=float
        )

        base_x = np.min(x_values)
        depth_diffs = x_values - base_x

        all_depth_diffs.extend(depth_diffs)

    return np.array(all_depth_diffs)

In [65]:
def classify_depth_by_delta(depth_diffs, delta):
    """
    depth_diff를 네 식에 따라 0,1,2,3으로 분류.
    """
    labels = np.zeros_like(depth_diffs, dtype=int)

    labels[(depth_diffs >= delta) & (depth_diffs < 2 * delta)] = 1
    labels[(depth_diffs >= 2 * delta) & (depth_diffs < 3 * delta)] = 2
    labels[depth_diffs >= 3 * delta] = 3

    return labels

In [66]:
def evaluate_delta(depth_diffs, delta):
    """
    특정 delta가 depth_diff를 얼마나 잘 나누는지 평가.
    낮을수록 좋음.
    """
    labels = classify_depth_by_delta(depth_diffs, delta)

    total_sse = 0

    for k in range(4):
        values = depth_diffs[labels == k]

        if len(values) == 0:
            # 빈 cluster가 생기는 delta는 좋지 않게 처리
            return np.inf

        center = values.mean()
        total_sse += np.sum((values - center) ** 2)

    return total_sse

In [67]:
def find_best_delta(depth_diffs, delta_min=0.5, delta_max=10.0, step=0.1):
    """
    여러 delta 후보 중 SSE가 가장 낮은 delta를 찾는다.
    """
    candidates = np.arange(delta_min, delta_max + step, step)

    results = []

    for delta in candidates:
        score = evaluate_delta(depth_diffs, delta)

        results.append({
            "delta": delta,
            "score": score
        })

    results_df = pd.DataFrame(results)

    best_row = results_df.loc[results_df["score"].idxmin()]
    best_delta = best_row["delta"]

    return best_delta, results_df

In [74]:
all_depth_diffs = []

for df in vertical_df:
    depth_diffs = collect_depth_diffs(df)
    all_depth_diffs.append(depth_diffs)

all_depth_diffs = np.concatenate(all_depth_diffs)

best_delta, delta_results = find_best_delta(
    all_depth_diffs,
    delta_min=0.5,
    delta_max=10.0,
    step=0.1
)

print("Best delta:", best_delta)

Best delta: 5.399999999999999


In [69]:
def classify_depth_level(depth_diff, delta):
    """
    depth_diff를 0, 1, 2, 3 level로 분류한다.

    0: depth_diff < delta
    1: delta <= depth_diff < 2*delta
    2: 2*delta <= depth_diff < 3*delta
    3: 3*delta <= depth_diff
    """
    if depth_diff < delta:
        return 0
    elif depth_diff < 2 * delta:
        return 1
    elif depth_diff < 3 * delta:
        return 2
    else:
        return 3

In [70]:
def add_depth_morphology_code(df, delta, defenders=DEFENDERS):
    df = df.copy()

    depth_diff_list = []
    depth_level_list = []
    depth_code_list = []

    for _, row in df.iterrows():
        order = row["y_order"]

        x_values = {
            p: row[f"{p}_x"]
            for p in defenders
        }

        base_x = min(x_values.values())

        depth_diffs = {
            p: x_values[p] - base_x
            for p in defenders
        }

        depth_levels = {
            p: classify_depth_level(depth_diffs[p], delta)
            for p in defenders
        }

        code = "".join(
            str(depth_levels[p])
            for p in order
        )

        depth_diff_list.append(depth_diffs)
        depth_level_list.append(depth_levels)
        depth_code_list.append(code)

    df["depth_diffs"] = depth_diff_list
    df["depth_levels"] = depth_level_list
    df["depth_morphology"] = depth_code_list

    return df

In [75]:
delta_dfs = []
for df in vertical_df:

    home_def_df = add_depth_morphology_code(
        df,
        delta=best_delta
    )
    delta_dfs.append(home_def_df)

delta_dfs[0][
    [
        "y_order",
        "depth_diffs",
        "depth_levels",
        "depth_morphology"
    ]
].head()

,y_order,depth_diffs,depth_levels,depth_morphology
0,"[RV, IVR, IVL, LV]","{'LV': 3.51, 'IVL': 0.0, 'IVR': 0.870000000000...","{'LV': 0, 'IVL': 0, 'IVR': 0, 'RV': 0}",0000
1,"[RV, IVR, IVL, LV]","{'LV': 3.469999999999999, 'IVL': 0.0, 'IVR': 0...","{'LV': 0, 'IVL': 0, 'IVR': 0, 'RV': 0}",0000
2,"[RV, IVR, IVL, LV]","{'LV': 3.4299999999999997, 'IVL': 0.0, 'IVR': ...","{'LV': 0, 'IVL': 0, 'IVR': 0, 'RV': 0}",0000
3,"[RV, IVR, IVL, LV]","{'LV': 3.4000000000000004, 'IVL': 0.0, 'IVR': ...","{'LV': 0, 'IVL': 0, 'IVR': 0, 'RV': 0}",0000
4,"[RV, IVR, IVL, LV]","{'LV': 3.3499999999999996, 'IVL': 0.0, 'IVR': ...","{'LV': 0, 'IVL': 0, 'IVR': 0, 'RV': 0}",0000


# 5. 합치기

In [78]:
for df in delta_dfs:

    df["full_morphology"] = (
        df["lat_morphology"]
        + "_"
        + df["depth_morphology"]
    )
delta_dfs[0][
    ["lat_morphology", "depth_morphology", "full_morphology"]
].head()

,lat_morphology,depth_morphology,full_morphology
0,111,0000,111_0000
1,111,0000,111_0000
2,111,0000,111_0000
3,111,0000,111_0000
4,111,0000,111_0000


In [80]:
for df in delta_dfs:
    depth_morph_count = df["depth_morphology"].nunique()

    print("Number of depth morphologies:", depth_morph_count)
    depth_morph_freq = (
    df["depth_morphology"]
    .value_counts()
    .reset_index()
    )

    depth_morph_freq.columns = ["depth_morphology", "count"]
    depth_morph_freq["ratio"] = depth_morph_freq["count"] / depth_morph_freq["count"].sum()

    print(depth_morph_freq)

Number of depth morphologies: 74
   depth_morphology  count     ratio
0              0000   6601  0.385347
1              1000   1938  0.113135
2              0001   1579  0.092177
3              1001    912  0.053240
4              1002    521  0.030414
..              ...    ...       ...
69             1302      4  0.000234
70             1200      4  0.000234
71             3030      2  0.000117
72             1103      2  0.000117
73             1021      1  0.000058

[74 rows x 3 columns]
Number of depth morphologies: 56
   depth_morphology  count     ratio
0              0000   7736  0.373467
1              0001   3428  0.165492
2              0002   1667  0.080477
3              1000   1416  0.068360
4              1001   1188  0.057353
5              2001    537  0.025924
6              0003    467  0.022545
7              1002    449  0.021676
8              2000    301  0.014531
9              1003    271  0.013083
10             3002    257  0.012407
11             2003    

In [81]:
all_delta_df = pd.concat(delta_dfs, ignore_index=True)

depth_morph_count = all_delta_df["depth_morphology"].nunique()
print("Total number of depth morphologies:", depth_morph_count)

depth_morph_freq_all = (
    all_delta_df["depth_morphology"]
    .value_counts()
    .reset_index()
)

depth_morph_freq_all.columns = ["depth_morphology", "count"]
depth_morph_freq_all["ratio"] = (
    depth_morph_freq_all["count"] / depth_morph_freq_all["count"].sum()
)

print(depth_morph_freq_all)

Total number of depth morphologies: 138
    depth_morphology  count     ratio
0               0000  66274  0.377554
1               0001  25088  0.142923
2               1000  16739  0.095360
3               1001   8010  0.045632
4               0002   7242  0.041257
..               ...    ...       ...
133             2310      7  0.000040
134             0311      6  0.000034
135             3012      5  0.000028
136             0212      4  0.000023
137             1022      3  0.000017

[138 rows x 3 columns]


# 6. Depth morphology 단순화

In [ ]:
# 15 simplify
def simplify_depth_morphology(code):
    """
    기존 depth morphology:
        0 = baseline
        1,2,3 = advanced 정도

    단순화:
        0 -> 0
        1,2,3 -> 1

    예:
        0000 -> 0000
        0001 -> 0001
        0002 -> 0001
        1002 -> 1001
        2030 -> 1010
    """
    return "".join("0" if c == "0" else "1" for c in str(code))

In [89]:
def simplify_depth_morphology_rank(code):
    """
    depth code의 상대적인 계단 구조를 보존한다.

    예:
        0000 -> 0000
        0002 -> 0001
        0003 -> 0001
        3110 -> 2110
        3120 -> 3120
        3030 -> 2020
    """
    values = [int(c) for c in str(code)]

    unique_values = sorted(set(values))

    rank_map = {
        value: rank
        for rank, value in enumerate(unique_values)
    }

    return "".join(str(rank_map[v]) for v in values)

In [83]:
all_delta_df["depth_morphology_16"] = (
    all_delta_df["depth_morphology"]
    .apply(simplify_depth_morphology)
)

In [84]:
print("Original depth morphologies:", all_delta_df["depth_morphology"].nunique())
print("Simplified depth morphologies:", all_delta_df["depth_morphology_16"].nunique())

Original depth morphologies: 138
Simplified depth morphologies: 15


In [87]:
depth_morph_16_freq = (
    all_delta_df["depth_morphology_16"]
    .value_counts()
    .reset_index()
)

depth_morph_16_freq.columns = ["depth_morphology_16", "count"]
depth_morph_16_freq["ratio"] = (
    depth_morph_16_freq["count"] / depth_morph_16_freq["count"].sum()
)
depth_morph_16_freq["percentage"] = depth_morph_16_freq["ratio"] * 100

depth_morph_16_freq

,depth_morphology_16,count,ratio,percentage
0,0000,66274,0.377554,37.755433
1,0001,34631,0.197288,19.728829
2,1001,26025,0.148261,14.826103
3,1000,21961,0.125109,12.510895
4,1101,5187,0.029550,2.954966
5,1011,3551,0.020230,2.022958
6,0010,3349,0.019079,1.907882
7,0100,3267,0.018612,1.861167
8,0011,2822,0.016077,1.607657
9,1010,2351,0.013393,1.339334


In [88]:
mapping_df = (
    all_delta_df[["depth_morphology", "depth_morphology_16"]]
    .drop_duplicates()
    .sort_values(["depth_morphology_16", "depth_morphology"])
    .reset_index(drop=True)
)

mapping_df

,depth_morphology,depth_morphology_16
0,0000,0000
1,0001,0001
2,0002,0001
3,0003,0001
4,0010,0010
...,...,...
133,3110,1110
134,3120,1110
135,3130,1110
136,3210,1110


In [91]:
all_delta_df["depth_morphology_rank"] = (
    all_delta_df["depth_morphology"]
    .apply(simplify_depth_morphology_rank)
)
mapping_df = (
    all_delta_df[
        ["depth_morphology", "depth_morphology_16", "depth_morphology_rank"]
    ]
    .drop_duplicates()
    .sort_values("depth_morphology")
    .reset_index(drop=True)
)

mapping_df.head(30)

,depth_morphology,depth_morphology_16,depth_morphology_rank
0,0000,0000,0000
1,0001,0001,0001
2,0002,0001,0001
3,0003,0001,0001
4,0010,0010,0010
5,0011,0011,0011
6,0012,0011,0012
7,0013,0011,0012
8,0020,0010,0010
9,0021,0011,0021


In [92]:
depth_rank_freq = (
    all_delta_df["depth_morphology_rank"]
    .value_counts()
    .reset_index()
)

depth_rank_freq.columns = ["depth_morphology_rank", "count"]
depth_rank_freq["ratio"] = (
    depth_rank_freq["count"] / depth_rank_freq["count"].sum()
)

depth_rank_freq

,depth_morphology_rank,count,ratio
0,0000,66274,0.377554
1,0001,34631,0.197288
2,1000,21961,0.125109
3,1001,11701,0.066659
4,1002,7534,0.042920
...,...,...,...
58,1120,11,0.000063
59,0123,9,0.000051
60,2310,7,0.000040
61,1210,7,0.000040
